# IAN 603 · Module 4
# Aggregating Data with PostgreSQL

Turn detailed rows into meaningful summaries with aggregates, grouping, conditional metrics, subtotals, and introductory windows.

Course module:  
https://ali2449.github.io/IAN-603-Preparing-Data-for-Analytics-/s_module4.html


## Learning goals

Work through the examples in order. Focus on the meaning of each row, the reason for each SQL clause, and how you would validate the result.


## 0. Install required Python packages

Run this only if the packages are not already installed.


In [ ]:
%pip install -q pandas sqlalchemy psycopg2-binary ipywidgets


## 1. Load the course database helper

This notebook uses a small course helper to open the PostgreSQL connection window and display query results.

**You only need this notebook file.** When you run the next cell, the helper is loaded automatically from the course GitHub repository if it is not already available.

If you would like your own copy of the helper, you can download it here:

[Download `ian603_database.py`](https://github.com/ali2449/IAN-603-Preparing-Data-for-Analytics-/raw/refs/heads/MSIA/ian603_database.py)

After the helper loads, continue to the connection section below.


In [ ]:
from pathlib import Path
import urllib.request

MODULE_FILE = Path("ian603_database.py")
MODULE_URL = (
    "https://raw.githubusercontent.com/"
    "ali2449/IAN-603-Preparing-Data-for-Analytics-/MSIA/"
    "ian603_database.py"
)

if not MODULE_FILE.exists():
    try:
        urllib.request.urlretrieve(MODULE_URL, MODULE_FILE)
        print("Course database helper downloaded.")
    except Exception as e:
        print("The database helper could not be downloaded automatically.")
        print("Use the download link in the Markdown cell above, then run this cell again.")
        raise

from ian603_database import IAN603Database

db = IAN603Database()

display_query = db.display_query
execute_query = db.execute_query
execute_script = db.execute_script

print("Database helper ready.")


## 2. Connect to PostgreSQL

Run the next cell to open the connection window. It works much like the connection settings you use in pgAdmin.

The usual local PostgreSQL defaults are already filled in:

| Setting | Default |
|---|---|
| Host | `localhost` |
| Database | `postgres` |
| Username | `postgres` |
| Port | `5432` |

Enter your PostgreSQL password, then click **Connect**.

If you are using your own computer, you may also choose to save the connection settings for the next time you open the notebook.


In [ ]:
db.show_connection_panel()


## 3. Check the connection

After you click **Connect**, run the next cell to make sure the notebook is ready to use PostgreSQL.


In [ ]:
if db.connected:
    db.test_connection()
else:
    print("No active PostgreSQL connection.")
    print("Return to the connection panel above and click Connect.")


## How to think about Module 4

Aggregation changes the **grain**.

Raw admissions:

| hadm_id | subject_id | diagnosis |
|---:|---:|---|
| 1001 | 1 | Pneumonia |
| 1002 | 1 | Fracture |
| 1003 | 2 | Pneumonia |

After grouping by diagnosis:

| diagnosis | count |
|---|---:|
| Pneumonia | 2 |
| Fracture | 1 |

The output is no longer one row per admission. It is:

> **one row per diagnosis**

### Key distinction

- `WHERE` filters **rows before grouping**
- `HAVING` filters **groups after aggregation**

Always state the desired output grain before writing `GROUP BY`.


# Teaching database

Run the next cell to create or reset the Module 4 teaching data. You can return to this cell whenever you want a clean starting point.


In [ ]:
setup_sql = "-- ============================================================\n-- IAN 603 - MODULE 4\n-- Aggregating Data with PostgreSQL\n-- ============================================================\n-- Companion file for s_module4.html\n-- PostgreSQL / pgAdmin Query Tool\n--\n-- Topics:\n-- COUNT, SUM, AVG, MIN, MAX, GROUP BY, HAVING, FILTER,\n-- CASE-based conditional aggregation, ROLLUP, CUBE,\n-- GROUPING SETS, and introductory window calculations.\n--\n-- Main habit: aggregation changes row grain. Define the desired\n-- grouping grain before writing the aggregate query.\n-- ============================================================\n\n-- ============================================================\n-- SECTION 1 - CREATE / RESET TEACHING DATABASE\n-- ============================================================\n\nDROP TABLE IF EXISTS grades,product_sales,employees,sales,admissions CASCADE;CREATE TABLE admissions(hadm_id integer PRIMARY KEY,subject_id integer NOT NULL,admit_time timestamp NOT NULL,discharge_time timestamp,diagnosis text NOT NULL);INSERT INTO admissions VALUES(1001,1,'2024-01-02 08:00','2024-01-05 12:00','Pneumonia'),(1002,1,'2024-03-10 09:00','2024-03-12 14:00','Fracture'),(1003,2,'2024-02-01 10:00','2024-02-04 10:00','Pneumonia'),(1004,2,'2024-04-18 12:00',NULL,'Pneumonia'),(1005,3,'2024-01-15 07:00','2024-01-16 18:00','Migraine'),(1006,3,'2024-05-06 11:00','2024-05-09 09:00','Fracture'),(1007,4,'2024-06-03 09:00','2024-06-08 09:00','Pneumonia'),(1008,5,'2024-06-14 13:00','2024-06-15 10:00','Asthma'),(1009,5,'2024-07-01 16:00','2024-07-04 11:00','Fracture'),(1010,6,'2024-07-20 06:00',NULL,'Pneumonia');CREATE TABLE sales(sale_id integer PRIMARY KEY,product text NOT NULL,quantity integer NOT NULL,price numeric(10,2) NOT NULL,category text NOT NULL,sale_date date NOT NULL);INSERT INTO sales VALUES(1,'Laptop',3,900,'Technology','2024-01-02'),(2,'Monitor',5,220,'Technology','2024-01-04'),(3,'Keyboard',6,45,'Technology','2024-01-05'),(4,'Desk',4,350,'Office','2024-01-08'),(5,'Chair',8,180,'Office','2024-01-09'),(6,'Notebook',12,8,'Office','2024-01-10'),(7,'Blender',5,95,'Home','2024-01-12'),(8,'Lamp',7,40,'Home','2024-01-14'),(9,'Storage Bin',10,22,'Home','2024-01-16'),(10,'Headphones',4,120,'Technology','2024-01-18');CREATE TABLE employees(employee_id integer PRIMARY KEY,name text NOT NULL,salary numeric(10,2) NOT NULL,department text NOT NULL,hire_date date NOT NULL);INSERT INTO employees VALUES(1,'Jordan Kim',92000,'Analytics','2021-04-12'),(2,'Taylor Brooks',81000,'Analytics','2022-07-01'),(3,'Casey Morgan',76000,'Analytics','2023-02-14'),(4,'Riley Chen',98000,'Engineering','2020-09-10'),(5,'Morgan Patel',88000,'Engineering','2022-11-20'),(6,'Jamie Rivera',72000,'Operations','2021-06-18'),(7,'Avery Scott',69000,'Operations','2023-03-25'),(8,'Cameron Bell',65000,'Operations','2024-01-15');CREATE TABLE product_sales(category text,product text,sales numeric(10,2));INSERT INTO product_sales VALUES('Technology','Laptop',2700),('Technology','Monitor',1100),('Technology','Keyboard',270),('Office','Desk',1400),('Office','Chair',1440),('Home','Blender',475),('Home','Lamp',280);CREATE TABLE grades(student_id integer,course text,grade numeric(5,2),semester text);INSERT INTO grades SELECT g,'IAN 603',72+(g%24),'Fall 2024' FROM generate_series(1,60) g;INSERT INTO grades SELECT 100+g,'IAN 601',70+(g%25),'Fall 2024' FROM generate_series(1,55) g;INSERT INTO grades SELECT 200+g,'IAN 620',74+(g%20),'Fall 2024' FROM generate_series(1,45) g;INSERT INTO grades SELECT 300+g,'IAN 603',75+(g%20),'Spring 2025' FROM generate_series(1,35) g;"

execute_script(setup_sql)


# BASIC AGGREGATION CHECKS


## BASIC AGGREGATION CHECKS · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT COUNT(*) AS admissions FROM admissions;"

display_query(query, title="BASIC AGGREGATION CHECKS · Step 1")


## BASIC AGGREGATION CHECKS · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT COUNT(*) AS sales_rows FROM sales;"

display_query(query, title="BASIC AGGREGATION CHECKS · Step 2")


## BASIC AGGREGATION CHECKS · Step 3

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT COUNT(*) AS employees FROM employees;"

display_query(query, title="BASIC AGGREGATION CHECKS · Step 3")


## BASIC AGGREGATION CHECKS · Step 4

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT COUNT(*) AS grade_rows FROM grades;"

display_query(query, title="BASIC AGGREGATION CHECKS · Step 4")


# RUNNABLE MODULE EXAMPLES


## Example 01

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    COUNT(*) AS all_admissions,\n    COUNT(discharge_time) AS completed_admissions,\n    COUNT(DISTINCT subject_id) AS unique_patients\nFROM admissions;"

display_query(query, title="Example 01")


## Example 02

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    COUNT(*) AS sale_rows,\n    SUM(quantity) AS units_sold,\n    ROUND(AVG(price), 2) AS average_price,\n    MIN(price) AS lowest_price,\n    MAX(price) AS highest_price\nFROM sales;"

display_query(query, title="Example 02")


## Example 03

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    ROUND((AVG(EXTRACT(EPOCH FROM (discharge_time - admit_time)) / 86400.0))::numeric, 2) AS average_stay_days,\n    COUNT(discharge_time) AS completed_admissions\nFROM admissions\nWHERE discharge_time IS NOT NULL;"

display_query(query, title="Example 03")


## Example 04

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    diagnosis,\n    COUNT(*) AS admission_count\nFROM admissions\nGROUP BY diagnosis\nORDER BY admission_count DESC, diagnosis;"

display_query(query, title="Example 04")


## Example 05

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    subject_id,\n    diagnosis,\n    COUNT(*) AS admission_count\nFROM admissions\nGROUP BY subject_id, diagnosis\nORDER BY subject_id, diagnosis;"

display_query(query, title="Example 05")


## Example 06

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    diagnosis,\n    subject_id,\n    COUNT(*) AS admission_count\nFROM admissions\nGROUP BY diagnosis;"

display_query(query, title="Example 06")


## Example 07

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    category,\n    SUM(quantity) AS units_sold,\n    ROUND(SUM(quantity * price), 2) AS category_revenue\nFROM sales\nWHERE price >= 25\nGROUP BY category\nHAVING SUM(quantity) > 10\nORDER BY category_revenue DESC;"

display_query(query, title="Example 07")


## Example 08

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    subject_id,\n    SUM(CASE WHEN diagnosis = 'Pneumonia' THEN 1 ELSE 0 END) AS pneumonia_admissions,\n    SUM(CASE WHEN diagnosis = 'Fracture' THEN 1 ELSE 0 END) AS fracture_admissions\nFROM admissions\nGROUP BY subject_id\nORDER BY subject_id;"

display_query(query, title="Example 08")


## Example 09

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    subject_id,\n    COUNT(*) FILTER (WHERE diagnosis = 'Pneumonia') AS pneumonia_admissions,\n    COUNT(*) FILTER (WHERE diagnosis = 'Fracture') AS fracture_admissions\nFROM admissions\nGROUP BY subject_id\nORDER BY subject_id;"

display_query(query, title="Example 09")


## Example 10

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    CASE WHEN GROUPING(category) = 1 THEN 'All categories' ELSE category END AS category_label,\n    CASE WHEN GROUPING(product) = 1 THEN 'All products' ELSE product END AS product_label,\n    SUM(sales) AS total_sales,\n    GROUPING(category, product) AS grouping_code\nFROM product_sales\nGROUP BY ROLLUP(category, product)\nORDER BY GROUPING(category), category, GROUPING(product), product;"

display_query(query, title="Example 10")


## Example 11

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    CASE WHEN GROUPING(category) = 1 THEN 'All categories' ELSE category END AS category_label,\n    CASE WHEN GROUPING(product) = 1 THEN 'All products' ELSE product END AS product_label,\n    SUM(sales) AS total_sales,\n    GROUPING(category, product) AS grouping_code\nFROM product_sales\nGROUP BY CUBE(category, product)\nORDER BY GROUPING(category), category, GROUPING(product), product;"

display_query(query, title="Example 11")


## Example 12

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    CASE WHEN GROUPING(category) = 1 THEN 'All categories' ELSE category END AS category_label,\n    CASE WHEN GROUPING(product) = 1 THEN 'All products' ELSE product END AS product_label,\n    SUM(sales) AS total_sales\nFROM product_sales\nGROUP BY GROUPING SETS ((category), (product), ())\nORDER BY GROUPING(category), category, GROUPING(product), product;"

display_query(query, title="Example 12")


## Example 13

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    name,\n    department,\n    salary,\n    RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS department_rank\nFROM employees\nORDER BY department, department_rank, name;"

display_query(query, title="Example 13")


## Example 14

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    name,\n    department,\n    salary,\n    ROUND(AVG(salary) OVER (PARTITION BY department), 2) AS department_average,\n    ROUND(AVG(salary) OVER (), 2) AS company_average\nFROM employees\nORDER BY department, salary DESC;"

display_query(query, title="Example 14")


## Example 15

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    sale_date,\n    product,\n    quantity * price AS row_revenue,\n    SUM(quantity * price) OVER (ORDER BY sale_date, sale_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue\nFROM sales\nORDER BY sale_date, sale_id;"

display_query(query, title="Example 15")


## Example 16

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    diagnosis,\n    COUNT(*) AS admission_count\nFROM admissions\nGROUP BY diagnosis\nHAVING COUNT(*) > 1\nORDER BY admission_count DESC, diagnosis;"

display_query(query, title="Example 16")


## Example 17

Run the SQL, inspect the output carefully, and explain what one output row represents.


In [ ]:
query = "SELECT\n    name,\n    department,\n    salary,\n    ROUND(AVG(salary) OVER (PARTITION BY department), 2) AS department_average,\n    ROUND(AVG(salary) OVER (), 2) AS company_average\nFROM employees\nORDER BY department, salary DESC;"

display_query(query, title="Example 17")


# VALIDATION PATTERNS


## VALIDATION PATTERNS · Step 1

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Compare raw row count with grouped row count.\nSELECT COUNT(*) AS source_rows FROM admissions;"

display_query(query, title="VALIDATION PATTERNS · Step 1")


## VALIDATION PATTERNS · Step 2

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT COUNT(*) AS diagnosis_groups\nFROM (\n    SELECT diagnosis\n    FROM admissions\n    GROUP BY diagnosis\n) AS g;"

display_query(query, title="VALIDATION PATTERNS · Step 2")


## VALIDATION PATTERNS · Step 3

Run this step and connect the result back to the module concept.


In [ ]:
query = "-- Compare source revenue with grouped revenue.\nSELECT SUM(quantity * price) AS source_revenue\nFROM sales;"

display_query(query, title="VALIDATION PATTERNS · Step 3")


## VALIDATION PATTERNS · Step 4

Run this step and connect the result back to the module concept.


In [ ]:
query = "SELECT SUM(category_revenue) AS grouped_revenue\nFROM (\n    SELECT category, SUM(quantity * price) AS category_revenue\n    FROM sales\n    GROUP BY category\n) AS x;"

display_query(query, title="VALIDATION PATTERNS · Step 4")


# Practice

1. Count all admissions, completed admissions, and unique patients.
2. Summarize admission count by diagnosis.
3. Calculate units and revenue by sales category.
4. Use FILTER to count Pneumonia and Fracture per subject.
5. Rank employees by salary within department.
6. Compare ROLLUP, CUBE, and GROUPING SETS outputs.
CHECKLIST
[ ] I can state the grouping grain.
[ ] I know WHERE filters rows before grouping.
[ ] I know HAVING filters groups after aggregation.
[ ] I understand COUNT(*) versus COUNT(column).
[ ] I validate totals after aggregation.

Try each task before looking back at the earlier examples.


In [ ]:
# Write your practice SQL here
practice_query = """

"""

# When ready:
# display_query(practice_query, title="Practice")


# Module 4 summary

Before moving on, make sure you can explain the main ideas in your own words and can describe the **grain** or row meaning of the important query results.

A good final check is:

1. What question did the query answer?
2. What did one input row represent?
3. What does one output row represent?
4. How did SQL change the data shape or meaning?
5. How would you validate that the result is correct?

## Next
Continue to **Module 5**.
